# Quantitative experiment: Online STDR time and memory

This notebook launches the time/memory experiment as a detached background process. Progress is written to a log file, so the run can continue after disconnecting from Jupyter as long as the remote server keeps normal user processes alive.

In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "Notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPT_PATH = PROJECT_ROOT / "Notebooks" / "run_quantitative_experiment_time_memory.py"
OUTPUT_ROOT = PROJECT_ROOT / "Results" / "Quantitative_experiment_time_memory"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SCRIPT_PATH

In [ ]:
# Selection controls. None means the full available dataset/all event types.
EVENT_TYPE_COUNT: int | None = None
MAX_EVENTS: int | None = None

# Initial experiment setup.
THETAS = (0.3,)
PROGRESS_EVERY = 100
SAVE_EVERY = 1000
RUN_ID = datetime.now().strftime("run_%Y%m%d_%H%M%S")
RUN_DIR = OUTPUT_ROOT / RUN_ID
LAUNCH_LOG = OUTPUT_ROOT / f"{RUN_ID}_nohup.out"

RUN_DIR, LAUNCH_LOG

In [ ]:
def optional_arg(command: list[str], flag: str, value: int | None) -> None:
    if value is not None:
        command.extend([flag, str(value)])


command = [
    sys.executable,
    str(SCRIPT_PATH),
    "--run-id",
    RUN_ID,
    "--progress-every",
    str(PROGRESS_EVERY),
    "--save-every",
    str(SAVE_EVERY),
]
optional_arg(command, "--event-type-count", EVENT_TYPE_COUNT)
optional_arg(command, "--max-events", MAX_EVENTS)
for theta in THETAS:
    command.extend(["--theta", str(theta)])

# Equivalent shell form for terminal use:
nohup_command = " ".join(["nohup", *command, ">", str(LAUNCH_LOG), "2>&1", "&"])
nohup_command

In [ ]:
# Launch the detached experiment. Progress goes to RUN_DIR / "progress.log".
# Re-running this cell starts a new run because RUN_ID was generated above.
with LAUNCH_LOG.open("ab") as launch_log:
    process = subprocess.Popen(
        ["nohup", *command],
        stdout=launch_log,
        stderr=subprocess.STDOUT,
        stdin=subprocess.DEVNULL,
        start_new_session=True,
        cwd=PROJECT_ROOT,
    )

print(f"Started PID: {process.pid}")
print(f"Run directory: {RUN_DIR}")
print(f"Progress log: {RUN_DIR / 'progress.log'}")
print(f"Launch log: {LAUNCH_LOG}")

In [ ]:
# Monitor progress after launch. Re-run this cell whenever you want to check status.
progress_log = RUN_DIR / "progress.log"
if progress_log.exists():
    lines = progress_log.read_text(encoding="utf-8").splitlines()
    print("\n".join(lines[-20:]))
else:
    print(f"Progress log not created yet: {progress_log}")

In [ ]:
# Load results after the background process finishes.
import pandas as pd

summary_path = RUN_DIR / "summary.csv"
measurements_path = RUN_DIR / "time_memory_measurements.csv"
plot_path = RUN_DIR / "plots" / "cumulative_time_memory.png"

if not summary_path.exists():
    raise FileNotFoundError(f"Run is not finished yet or summary is missing: {summary_path}")

summary_df = pd.read_csv(summary_path)
measurements_df = pd.read_csv(measurements_path)
print(f"Plot: {plot_path}")
summary_df